# Fine-tune FLAN-T5 for natural, reason-aware restaurant reports

Upload only `aspect_reasons_restaurant.json`. The real `predicted` table is reserved for final evaluation. Training uses synthetic aspect tables, varied reason evidence, shuffled row order, and several discourse styles. Unlike the old notebook, inputs contain no preassigned roles and targets are not one rigid four-sentence template.

In [1]:
!pip install -q -U transformers datasets accelerate sentencepiece
import torch, transformers
print('torch:', torch.__version__, '| transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable Runtime > Change runtime type > T4 GPU'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.9 MB/s eta 0:00:00
torch: 2.11.0+cu128 | transformers: 5.14.1
CUDA: True | GPU: Tesla T4


In [2]:
from google.colab import files
import json
uploaded = files.upload()
stats_name = next((name for name in uploaded if name.endswith(('.json', '.txt'))), None)
assert stats_name, 'Upload output/aspect_reasons_restaurant.json'
payload = json.loads(uploaded[stats_name].decode('utf-8'))
real_rows = payload['predicted']
assert real_rows and all(r['positive'] + r['negative'] + r['neutral'] == r['total'] for r in real_rows)
real_aspects = {r['aspect'].strip().lower() for r in real_rows}
print('Loaded', len(real_rows), 'real predicted aspects for FINAL EVALUATION ONLY')

Saving aspect_reasons_restaurant.json to aspect_reasons_restaurant.json
Loaded 585 real predicted aspects for FINAL EVALUATION ONLY


In [3]:
import random
from datasets import Dataset

POS_REASONS = ['welcoming','flavorful','attentive','affordable','fresh','comfortable','creative','generous','reliable','pleasant','delightful','efficient']
NEG_REASONS = ['slow','bland','costly','rude','noisy','cramped','cold','inconsistent','limited','stale','careless','disappointing']
NEU_REASONS = ['average','standard','ordinary','simple','adequate','typical']
SYL = ['al','bor','cen','dor','el','fin','gal','hor','ian','jor','kel','lor','mor','nel','or','pra','quil','rin','sor','tor','ul','ven','wel','yor','zen']

def synthetic_name(i):
    base = SYL[i % len(SYL)] + SYL[(i // len(SYL)) % len(SYL)] + SYL[(i // (len(SYL)**2)) % len(SYL)]
    return [base, f'{base} counter', f'{base}-area', f'{base} selection'][i % 4]

names = []
i = 0
while len(names) < 360:
    name = synthetic_name(i).lower(); i += 1
    if name not in real_aspects and name not in names: names.append(name)
train_names, valid_names, test_names = names[:280], names[280:320], names[320:]
assert set(train_names).isdisjoint(valid_names) and set(train_names).isdisjoint(test_names)
assert set(valid_names).isdisjoint(test_names) and set(names).isdisjoint(real_aspects)

def reason_pairs(rng, vocabulary, sentiment_count):
    words = rng.sample(vocabulary, 2)
    high = max(2, min(sentiment_count, rng.randint(5, max(5, sentiment_count // 2))))
    low = max(1, min(high - 1, rng.randint(1, max(1, high - 1))))
    return [[words[0], high], [words[1], low]]

def make_row(rng, aspect, role):
    if role == 'attention':
        total = rng.randint(120, 180); neutral = rng.randint(5, 15); positive = rng.randint(65, total-neutral-30); negative = total-positive-neutral
    elif role == 'strength':
        total = rng.randint(55, 100); positive = rng.randint(int(total*.78), int(total*.91)); negative = rng.randint(2, total-positive); neutral = total-positive-negative
    elif role == 'concern':
        total = rng.randint(55, 100); negative = rng.randint(int(total*.65), int(total*.85)); positive = rng.randint(2, total-negative); neutral = total-positive-negative
    else:
        total = rng.randint(55, 100); neutral = rng.randint(3, 12); positive = (total-neutral)//2 + rng.randint(-3, 3); negative = total-positive-neutral
    return {'aspect':aspect,'total':total,'positive':positive,'negative':negative,'neutral':neutral,
            'positive_reasons':reason_pairs(rng,POS_REASONS,positive),'negative_reasons':reason_pairs(rng,NEG_REASONS,negative),
            'neutral_reasons':reason_pairs(rng,NEU_REASONS,max(neutral,2)),'role':role}

def build_input(rows):
    lines=[]
    for r in rows:
        pr=', '.join(f'{x}:{n}' for x,n in r['positive_reasons']); nr=', '.join(f'{x}:{n}' for x,n in r['negative_reasons'])
        lines.append(f"aspect={r['aspect']} | total={r['total']} | positive={r['positive']} | negative={r['negative']} | neutral={r['neutral']} | positive reasons={pr} | negative reasons={nr}")
    return ('Write a natural analytical restaurant-feedback paragraph. Compare the aspects, explain strengths and concerns with supplied reasons, cite exact counts, and invent nothing. Avoid a row-by-row list.\nDATA:\n' + '\n'.join(lines) + '\nREPORT:')

def render_target(rows, style):
    by = {r['role']:r for r in rows}; a,b,c,d = by['attention'],by['strength'],by['concern'],by['mixed']
    ap, bp, cn, dp = a['positive_reasons'][0][0], b['positive_reasons'][0][0], c['negative_reasons'][0][0], d['positive_reasons'][0][0]
    dn = d['negative_reasons'][0][0]
    if style == 0:
        return (f"Overall, {a['aspect']} dominated the conversation with {a['total']} mentions, including {a['positive']} positive responses, often describing it as {ap}. "
                f"The clearest strength was {b['aspect']}: {b['positive']} of its {b['total']} mentions were positive, led by comments such as {bp}. "
                f"By contrast, {c['aspect']} was the main concern, with {c['negative']} negative mentions out of {c['total']}, most commonly linked to {cn}. "
                f"Views on {d['aspect']} were more divided; among {d['total']} mentions, {d['positive']} were positive and {d['negative']} negative, reflecting both {dp} and {dn} experiences.")
    if style == 1:
        return (f"Customer feedback points to {b['aspect']} as a standout: it drew {b['positive']} positive comments from {b['total']} mentions, frequently because it was {bp}. "
                f"Although {a['aspect']} attracted the most discussion at {a['total']} mentions and earned {a['positive']} positive reactions associated with {ap}, not every area performed as well. "
                f"In particular, {c['aspect']} generated {c['negative']} negative responses among {c['total']} mentions, with {cn} recurring as the chief complaint. "
                f"Meanwhile, {d['aspect']} split opinion: its {d['total']} mentions contained {d['positive']} positive and {d['negative']} negative reactions, commonly framed as either {dp} or {dn}.")
    if style == 2:
        return (f"The strongest takeaway is the contrast between {b['aspect']} and {c['aspect']}. {b['aspect']} was praised in {b['positive']} of {b['total']} mentions, especially for being {bp}, whereas {c['aspect']} received {c['negative']} negative comments out of {c['total']}, largely citing {cn}. "
                f"At the same time, {a['aspect']} remained the most visible topic, appearing {a['total']} times with {a['positive']} positive mentions and {ap} as a recurring reason. "
                f"Feedback on {d['aspect']} was less settled: {d['positive']} positive versus {d['negative']} negative reactions across {d['total']} mentions, with experiences ranging from {dp} to {dn}.")
    return (f"Among the areas discussed, {a['aspect']} had the broadest reach, accounting for {a['total']} mentions; {a['positive']} were positive and frequently emphasized {ap}. "
            f"More decisively, {b['aspect']} emerged as a strength, as {b['positive']} of {b['total']} mentions praised qualities such as {bp}. "
            f"The priority for improvement is {c['aspect']}, where {c['negative']} of {c['total']} mentions were negative and {cn} was the leading explanation. "
            f"Finally, {d['aspect']} produced a mixed picture, combining {d['positive']} positive and {d['negative']} negative reactions in {d['total']} mentions, particularly around {dp} and {dn}.")

def make_examples(count, seed, pool):
    rng=random.Random(seed); result=[]
    for i in range(count):
        chosen=rng.sample(pool,4); rows=[make_row(rng,name,role) for name,role in zip(chosen,['attention','strength','concern','mixed'])]
        target=render_target(rows,i%4); rng.shuffle(rows); result.append({'input':build_input(rows),'target':target,'rows':json.dumps(rows)})
    return result

train_raw=Dataset.from_list(make_examples(4000,42,train_names)); valid_raw=Dataset.from_list(make_examples(400,43,valid_names)); test_raw=Dataset.from_list(make_examples(300,44,test_names))
print('Leakage audit passed. Sizes:',len(train_raw),len(valid_raw),len(test_raw)); print(train_raw[0]['input']); print('\nTARGET:\n',train_raw[0]['target'])

Leakage audit passed. Sizes: 4000 400 300
Write a natural analytical restaurant-feedback paragraph. Compare the aspects, explain strengths and concerns with supplied reasons, cite exact counts, and invent nothing. Avoid a row-by-row list.
DATA:
aspect=horcenal counter | total=134 | positive=71 | negative=56 | neutral=7 | positive reasons=delightful:7, reliable:5 | negative reasons=cold:5, slow:1
aspect=prafinal | total=61 | positive=14 | negative=40 | neutral=7 | positive reasons=flavorful:6, comfortable:5 | negative reasons=noisy:19, slow:18
aspect=moralal | total=100 | positive=88 | negative=10 | neutral=2 | positive reasons=creative:33, affordable:18 | negative reasons=slow:5, costly:3
aspect=alfinal counter | total=95 | positive=44 | negative=39 | neutral=12 | positive reasons=comfortable:11, pleasant:2 | negative reasons=slow:8, careless:7
REPORT:

TARGET:
 Overall, horcenal counter dominated the conversation with 134 mentions, including 71 positive responses, often describing it 

In [4]:
import gc, math, shutil
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, get_linear_schedule_with_warmup
from transformers.optimization import Adafactor

MODEL_NAME='google/flan-t5-base'; OUTPUT_DIR='/content/flan-t5-reasoned-report-model'; BEST=OUTPUT_DIR+'/best'
shutil.rmtree(OUTPUT_DIR,ignore_errors=True); gc.collect(); torch.cuda.empty_cache(); device=torch.device('cuda')
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    model_inputs=tokenizer(batch['input'],max_length=384,truncation=True)
    model_inputs['labels']=tokenizer(text_target=batch['target'],max_length=220,truncation=True)['input_ids']; return model_inputs
train_ds=train_raw.map(tokenize,batched=True,remove_columns=train_raw.column_names); valid_ds=valid_raw.map(tokenize,batched=True,remove_columns=valid_raw.column_names)
model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device); model.gradient_checkpointing_enable(); model.config.use_cache=False
collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model); train_loader=DataLoader(train_ds,batch_size=2,shuffle=True,collate_fn=collator); valid_loader=DataLoader(valid_ds,batch_size=2,collate_fn=collator)
EPOCHS,ACCUM,LR,PATIENCE=4,8,5e-5,2
optimizer=Adafactor(model.parameters(),lr=LR,scale_parameter=False,relative_step=False,warmup_init=False)
updates=math.ceil(len(train_loader)/ACCUM)*EPOCHS; scheduler=get_linear_schedule_with_warmup(optimizer,int(updates*.05),updates)
def move(batch): return {k:v.to(device) for k,v in batch.items()}
@torch.no_grad()
def val_loss():
    model.eval(); values=[]
    for batch in tqdm(valid_loader,desc='Validation',leave=False):
        loss=model(**move(batch)).loss
        if not torch.isfinite(loss): raise RuntimeError('Non-finite validation loss')
        values.append(loss.item())
    return sum(values)/len(values)
best=float('inf'); stale=0; history=[]
for epoch in range(1,EPOCHS+1):
    model.train(); optimizer.zero_grad(set_to_none=True); losses=[]
    for step,batch in tqdm(enumerate(train_loader),total=len(train_loader),desc=f'Epoch {epoch}/{EPOCHS}'):
        loss=model(**move(batch)).loss
        if not torch.isfinite(loss): raise RuntimeError(f'Non-finite loss at {epoch}/{step}')
        losses.append(loss.item()); (loss/ACCUM).backward()
        if (step+1)%ACCUM==0 or step+1==len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(),.5); optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
    train_loss=sum(losses)/len(losses); validation_loss=val_loss(); history.append({'epoch':epoch,'train':train_loss,'validation':validation_loss})
    print(history[-1])
    if validation_loss < best:
        best=validation_loss; stale=0; shutil.rmtree(BEST,ignore_errors=True); model.save_pretrained(BEST); tokenizer.save_pretrained(BEST)
    else:
        stale+=1
        if stale>=PATIENCE: break
del model,optimizer,scheduler; gc.collect(); torch.cuda.empty_cache(); model=AutoModelForSeq2SeqLM.from_pretrained(BEST).to(device).eval()
print('Training complete. Best validation loss:',best)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Validation:   0%|          | 0/200 [00:00<?, ?it/s]

{'epoch': 4, 'train': 0.019618490361142903, 'validation': 0.021724338941276074}


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Training complete. Best validation loss: 0.01997602023649961


In [5]:
import re
def generate(texts,batch_size=4):
    outputs=[]
    for start in tqdm(range(0,len(texts),batch_size),desc='Generating'):
        encoded=tokenizer(texts[start:start+batch_size],return_tensors='pt',padding=True,truncation=True,max_length=384).to(device)
        ids=model.generate(**encoded,max_new_tokens=220,num_beams=4,do_sample=False,no_repeat_ngram_size=0)
        outputs.extend(tokenizer.batch_decode(ids,skip_special_tokens=True))
    return outputs

def grounded(report, rows):
    text=report.lower(); details=[]
    for row in rows:
        aspect=row['aspect'].lower(); present=bool(re.search(rf'(?<!\\w){re.escape(aspect)}(?!\\w)',text))
        allowed={row[k] for k in ['total','positive','negative','neutral']}
        allowed.update(n for field in ['positive_reasons','negative_reasons','neutral_reasons'] for _,n in row[field])
        reasons=[reason for field in ['positive_reasons','negative_reasons','neutral_reasons'] for reason,_ in row[field] if reason in text]
        valid=present and str(row['total']) in text and bool(reasons)
        details.append({'aspect':aspect,'valid':valid,'reasons':reasons})
    all_allowed={n for row in rows for n in [row['total'],row['positive'],row['negative'],row['neutral']]}
    all_allowed.update(n for row in rows for field in ['positive_reasons','negative_reasons','neutral_reasons'] for _,n in row[field])
    unsupported=[int(n) for n in re.findall(r'\d+',report) if int(n) not in all_allowed]
    return {'passed':all(x['valid'] for x in details) and not unsupported,'checks':details,'unsupported_numbers':unsupported}

test_reports=generate(list(test_raw['input'])); test_rows=[json.loads(value) for value in test_raw['rows']]
test_checks=[grounded(report,rows) for report,rows in zip(test_reports,test_rows)]
print(f"Synthetic grounded pass rate: {sum(x['passed'] for x in test_checks)}/{len(test_checks)} = {sum(x['passed'] for x in test_checks)/len(test_checks):.2%}")

def select_real(rows):
    candidates=sorted([r for r in rows if r['total']>=10],key=lambda r:(-r['total'],r['aspect'])); pool=candidates[:30]; selected=[candidates[0]]
    remaining=lambda:[r for r in pool if r['aspect'] not in {x['aspect'] for x in selected}]
    selected.append(max(remaining(),key=lambda r:(r['positive']/r['total'],r['total']))); selected.append(max(remaining(),key=lambda r:(r['negative']/r['total'],r['total']))); selected.append(min(remaining(),key=lambda r:(abs(r['positive']-r['negative'])/r['total'],-r['total'])))
    return selected
def real_input(rows):
    clean=[]
    for row in rows:
        item=dict(row); item['role']='unused'; clean.append(item)
    return build_input(clean)
selected=select_real(real_rows); report=generate([real_input(selected)],1)[0]; check=grounded(report,selected)
print('SELECTED:',[(r['aspect'],r['total']) for r in selected]); print('\nFLAN-T5 REPORT:\n',report); print('\nGROUNDED:',check)
with open('/content/flan_t5_reasoned_evaluation.json','w',encoding='utf-8') as f: json.dump({'selected':selected,'report':report,'factual_check':check},f,ensure_ascii=False,indent=2)

Generating:   0%|          | 0/75 [00:00<?, ?it/s]

Synthetic grounded pass rate: 234/300 = 78.00%


Generating:   0%|          | 0/1 [00:00<?, ?it/s]

SELECTED: [('food', 827), ('indian food', 22), ('dessert', 26), ('waiter', 42)]

FLAN-T5 REPORT:
 Customer feedback points to waiter as a standout: it drew 24 positive comments from 42 mentions, frequently because he was attentive. Although food attracted the most discussion at 827 mentions and earned 621 positive reactions associated with great, not every area performed as well. In particular, Indian food generated 164 negative responses among 22 mentions, with not inspired recurring as the chief complaint. Meanwhile, dessert split opinion: its 26 mentions contained 9 positive and 16 negative reactions, commonly framed as either divine or not inspired.

GROUNDED: {'passed': True, 'checks': [{'aspect': 'food', 'valid': True, 'reasons': ['great']}, {'aspect': 'indian food', 'valid': True, 'reasons': ['great']}, {'aspect': 'dessert', 'valid': True, 'reasons': ['divine', 'not inspired', 'so']}, {'aspect': 'waiter', 'valid': True, 'reasons': ['attentive']}], 'unsupported_numbers': []}


In [6]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/flan-t5-reasoned-report-model','zip',BEST)
files.download('/content/flan_t5_reasoned_evaluation.json'); files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>